In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from collections import Counter
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

# Load the dataset
data_path= "C:/Users/olufe/projects/Journal/dataset/HomeA_unsupervised_sim_combined_shuffled.csv"
data_df = pd.read_csv(data_path)

# Extract features (X) and target labels (y) from the loaded data
X = data_df.drop(columns=["Label"])  # Replace "target_column_name" with the actual column name for the target variable
y = data_df["Label"]  # Replace "target_column_name" with the actual column name for the target variable

# Print the extracted values of X and y
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (35330, 1380)
y shape: (35330,)


In [7]:
# Convert data to numpy arrays and normalize the data to float32
# data = data_df.values.astype('float32')
# data = (data - data.min()) / (data.max() - data.min())

# data = X.values.astype('float32')
# data = (data - data.min()) / (data.max() - data.min())

#import numpy as np

X_array = X.values  # Convert X (DataFrame) to a 2D NumPy array
y_array = y.values  # Convert y (Series) to a 1D NumPy array

# Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_array)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_array, test_size=0.2, random_state=42)

In [14]:
# summarize the new class distribution
counter = Counter(y)
print(counter)

Counter({0: 35186, 1: 144})


In [15]:
# Define the MLP Autoencoder model
input_dim = data.shape[1]
encoded_dim = int(input_dim / 2)

model = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    keras.layers.Dense(encoded_dim, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(input_dim, activation='sigmoid')
])

In [16]:
model.layers

In [17]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 64)                88448     
                                                                 
 dense_5 (Dense)             (None, 690)               44850     
                                                                 
 dense_6 (Dense)             (None, 64)                44224     
                                                                 
 dense_7 (Dense)             (None, 1381)              89765     
                                                                 
Total params: 267,287
Trainable params: 267,287
Non-trainable params: 0
_________________________________________________________________


In [18]:
# Compile the model
model.compile(optimizer='adam', loss='mse')

# Function to calculate performance metrics
def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tnr = tn / (tn + fp)
    fpr = fp / (tn + fp)
    fnr = fn / (fn + tp)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * (precision * recall) / (precision + recall)
    auc = roc_auc_score(y_true, y_pred)
    return tnr, fpr, fnr, precision, recall, f1, auc

In [20]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.base import clone

# skfolds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# for train_index, test_index in skfolds.split(X_train, y_train):
#     clone_model = clone(model)
#     X_train_folds = X_train[train_index]
#     y_train_folds = y_train[train_index]
#     X_test_fold = X_train[test_index]
#     y_test_fold = y_train[test_index]

#     clone_clf.fit(X_train_folds, y_train_folds)
#     y_pred = clone_model.predict(X_test_fold)
#     n_correct = sum(y_pred == y_test_fold)
#     print(n_correct / len(y_pred))

In [1]:
# Perform k-fold cross-validation
kfold = KFold(n_splits=3, shuffle=True, random_state=42)

all_metrics = []
for train_idx, test_idx in kfold.split(data):
    data_train_fold, data_val_fold = data[train_idx], data[test_idx]

    # Train the model
    model.fit(data_train_fold, data_train_fold, epochs=50, batch_size=32, verbose=0)

    # Make predictions on the validation set
    val_predictions = model.predict(data_val_fold)

    # Binarize predictions using a threshold (e.g., 0.5)
    threshold = 0.5
    # threshold = 0
    val_predictions_binarized = (val_predictions > threshold).astype(int)

    # Calculate performance metrics for this fold
    tnr, fpr, fnr, precision, recall, f1, auc = calculate_metrics(data_val_fold.flatten(), val_predictions_binarized.flatten())

    all_metrics.append((tnr, fpr, fnr, precision, recall, f1, auc))

In [ ]:
# Calculate average performance metrics over all folds
mean_metrics = np.mean(all_metrics, axis=0)

# Print the results
print("Average TNR: ", mean_metrics[0])
print("Average FPR: ", mean_metrics[1])
print("Average FNR: ", mean_metrics[2])
print("Average Precision: ", mean_metrics[3])
print("Average Recall: ", mean_metrics[4])
print("Average F1-score: ", mean_metrics[5])
print("Average AUC: ", mean_metrics[6])

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, confusion_matrix

# ... Load and preprocess your data ...

# Define the MLP Autoencoder model

# Compile the model

# Function to calculate performance metrics
def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tnr = tn / (tn + fp)
    fpr = fp / (tn + fp)
    fnr = fn / (fn + tp)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * (precision * recall) / (precision + recall)
    auc = roc_auc_score(y_true, y_pred)
    return tnr, fpr, fnr, precision, recall, f1, auc

# Perform k-fold cross-validation

# ... Training and validation ...

# Loop over folds
all_metrics = []
for train_idx, test_idx in kfold.split(data):
    data_train_fold, data_val_fold = data[train_idx], data[test_idx]
    val_predictions = model.predict(data_val_fold)
    
    # Binarize predictions using a threshold (e.g., 0.5)
    threshold = 0.5  # Adjust the threshold as needed
    val_predictions_binarized = (val_predictions > threshold).astype(int)
    
    # Calculate performance metrics for this fold
    tnr, fpr, fnr, precision, recall, f1, auc = calculate_metrics(data_val_fold.flatten(), val_predictions_binarized.flatten())
    all_metrics.append((tnr, fpr, fnr, precision, recall, f1, auc))

# ... Calculate and print average metrics ...
